# Urban and rural surrounding areas selection

***
> This work is licensed under a [Creative Commons Attribution 4.0 International License](http://creativecommons.org/licenses/by/4.0)
>
> ![Creative Commons License >](https://i.creativecommons.org/l/by/4.0/88x31.png)

Adapted from the URCLIMASK example notebook (https://github.com/FPS-URB-RCC/URCLIMASK, CC BY 4.0).
The URCLIMASK code was adapted to read simulation data on a Lambert conformal grid and our own urban station data.

**Role in the paper.** For one city, this notebook computes the six p2p and a2a evaluation methods for the
nighttime UHI (UHIN = urban minus rural daily minimum temperature, monthly means):
1. builds the URCLIMASK urban/rural masks (a2a) from the static model fields;
2. classifies the stations as urban/rural with the urban centre (UCDB) and with LCZ;
3. computes simulated and observed UHIN per method, adds the p2a result of
   `method_2_blended_approach/UHI_blended_approach_<City>.ipynb` (run that notebook first) and saves all seven methods to
   `results/data/<City>/UHI_<City>_annualcycle_method_comparison_<var>.{nc,csv}`.

| Code name (columns / variables) | Paper name |
|---|---|
| `p2p_un_ucdb` | p2p_UCN_uc |
| `p2p_un_lcz` | p2p_UCN_lcz |
| `p2p_ms_ucdb` | p2p_MS_uc |
| `urclim_un_ucdb` | a2a_UCN_uc |
| `urclim_un_lcz` | a2a_UCN_lcz |
| `urclim_ms_ucdb` | a2a_MS_uc |
| `blend_ms_ucdb` (method 2, blended) | p2a_MS_dist |

## Load libraries

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import glob
import numpy as np
import xarray as xr
import cf_xarray as cfxr
from pathlib import Path
import metobs_toolkit

from rasterio import warp

import cftime
import pandas as pd
import matplotlib as plt
from pprint import pprint
import importlib
import cartopy.crs as ccrs
import geopandas as gpd
import rasterio
import sys

from urclimask.urban_areas import (
    UrbanVicinity
)
from urclimask.utils import (
    kelvin2degC,
    traverseDir,
    fix_360_longitudes,
    RCM_DICT,
    load_ucdb_city,
    align_to_reference_grid,
    classify_stations_from_ucdb_city,
    plot_UHI_p2p_cycle,
    compute_UHI_p2p, 
    from_latlon_to_xy,
    classify_stations_from_LCZ
)

from urclimask.UHI_analysis import (
    UrbanIsland
)
from urclimask.GHCNd_stations import (
    load_ghcnd_stations,
    get_ghcnd_df,
    get_valid_timeseries, 
    available_vars,
    inside_city
)
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


# Urclimask

## Urclimask settings

In [ ]:
city = 'Rennes'
lon_city =  -1.677997294481546
lat_city = 48.11751888591575

start = "2018-01-01"
stop = "2018-12-31"

network_og = network = 'city_network'
subset_og = subset = '_gapfree'

#Ghent (51.05454277446639, 3.7249005076743438)
domain = 'EUROCITIES'
domain_resolution = 2.5

model = 'ALAROSFX'
scenario = "evaluation" 

urban_var = 'sfturf'

urban_th = 40
urban_sur_th = 10
orog_diff = 100
sftlf_th = 60
ratio_r2u = 2
min_city_size = 5
lon_lim = 0.2
lat_lim = 0.2

Create a directory to save the results for this particular city and model

In [ ]:
ROOT = Path.cwd().parent.parent.parent
data_dir =  ROOT / f'data/'
plot_dir =  ROOT / f'results/figures/{city}/method_2_urclimask_{network}'
os.makedirs(plot_dir, exist_ok=True)
save_masks_dir = ROOT / f'data/metadata/masks/'
os.makedirs(save_masks_dir, exist_ok=True)
save_dir =  ROOT / f'results/data/{city}/'
os.makedirs(save_dir, exist_ok=True)
base_filename = f'{city}-{domain}_ECMWF-ERA5_{scenario}_r1i1p1f1_{network}_{subset}'

## Define urban areas and vicinity

### Load city shapefile and static variables
Load static variables, such as urban fraction (sfturf), terrain elevation (orography) or land-sea fraction (sftlf).

In [ ]:
ucdb_city = load_ucdb_city(f'{data_dir}/metadata/UCDB_shapefiles', city)

static_input_path = f'{data_dir}/metadata/simulations/static_variables_eurocities.nc'
static_input = xr.open_dataset(static_input_path,engine='netcdf4')
static_input = fix_360_longitudes(static_input)

ds_sfturf, ds_orog, ds_sftlf = static_input, static_input, static_input


In [ ]:
#Add parameters to the function
URBAN = UrbanVicinity(
    urban_sur_th = urban_sur_th,
    orog_diff = orog_diff,
    sftlf_th = sftlf_th,
    ratio_r2u = ratio_r2u,
    min_city_size = min_city_size,
    lon_city = lon_city,
    lat_city = lat_city,
    lon_lim = lon_lim,
    lat_lim = lat_lim,
    model = model,
    domain = domain,
    urban_th = urban_th,
    urban_var = urban_var
)

#crop around the city
ds_sfturf = URBAN.crop_area_city(ds = ds_sfturf, res = domain_resolution)
ds_orog = URBAN.crop_area_city(ds = ds_orog, res = domain_resolution)
ds_sftlf = URBAN.crop_area_city(ds = ds_sftlf, res = domain_resolution)

#Define masks using the parameters above
sfturf_mask, sfturf_sur_mask, orog_mask, sftlf_mask = URBAN.define_masks(
    ds_sfturf = ds_sfturf, 
    ds_orog = ds_orog, 
    ds_sftlf = ds_sftlf,
)

### Define rural vicinity areas

We use an iterative algorithm to select the urban vicinity excluding masked areas

In [ ]:
urmask = URBAN.select_urban_vicinity(
    sfturf_mask = sfturf_mask, 
    orog_mask = orog_mask,
    sftlf_mask = sftlf_mask,
    sfturf_sur_mask = sfturf_sur_mask,
    debug = True,
    max_iterations=200,
)

We can plot the masks and the urban vs vicinity areas. The latter are are passed via the `urban_areas` parameter. They are hidden if set to `None`.

In [ ]:
fig = URBAN.plot_static_variables(ds_sfturf = ds_sfturf, 
                                  ds_orog = ds_orog, 
                                  ds_sftlf = ds_sftlf,
                                  sfturf_mask = sfturf_mask, 
                                  orog_mask = orog_mask, 
                                  sftlf_mask = sftlf_mask,
                                  urban_areas = urmask,
                                  ucdb_city = ucdb_city,
                                 composite = False) 
fig.savefig(f"{plot_dir}/urmask_{base_filename}_fx_.pdf", bbox_inches='tight')

In [ ]:
fig = URBAN.plot_static_variables(ds_sfturf = ds_sfturf, 
                                  ds_orog = ds_orog, 
                                  ds_sftlf = ds_sftlf,
                                  sfturf_mask = sfturf_mask, 
                                  orog_mask = orog_mask, 
                                  sftlf_mask = sftlf_mask,
                                  urban_areas = urmask,
                                  ucdb_city = ucdb_city,
                                 composite = True, )
fig.savefig(f"{plot_dir}/urmask_{base_filename}_fx.pdf", bbox_inches='tight')

Save these masks and urban fraction to NetCDF

In [ ]:

urmask.to_netcdf(f"{save_masks_dir}/urmask_{base_filename}_fx.nc")

## Load simulation data

In [ ]:
var='tasmin'


In [ ]:
path_monmean = f"{data_dir}/processed/simulations/monthly_means/{domain}_operational_cy43_{var}_monmean_*.nc"
da_var = xr.open_mfdataset(sorted(glob.glob(path_monmean)), engine="netcdf4", chunks="auto").rename({'tas':var}).sel(time=slice(start,stop)) 

ds_RCM_urb = URBAN.crop_area_city(ds = da_var, res = domain_resolution)
ds_RCM_urb["time"] = ds_RCM_urb.indexes["time"].to_period("M").to_timestamp()
ds_RCM_urb

In [ ]:
# Align static information to model data coordinates (in case their is a rounding error in the grid coordinates causing a mismatch in the grid)
urmask, alignment_status = align_to_reference_grid(urmask, ds_RCM_urb, coord_names=['x', 'y'])
print(alignment_status)

## Load Observational data

### Load urban climate network data

In [ ]:
start_o = start #because 2020, the year represented by the Urban network is not in the meteostations network
stop_o = stop

# load station observation data (preproccessed: monthly mean of daily data)
obs_path = f"{data_dir}/processed/observations/{city}/{network}/datasets/{subset}/{city}_{network}{subset}_monthlymean_daily{var[3:]}.nc"
ds_obs = xr.open_dataset(obs_path, engine="netcdf4", chunks="auto").sel(time=slice(start_o,stop_o))
obs_timeseries = ds_obs.to_dataframe().copy()

stations = obs_timeseries.columns.to_list()
print(f"{len(stations)} available stations: {stations}")

Prepare obs_attributes metadata for two urban/rural classfiction approaches: core based (ucdb polygon) and LCZ based

In [ ]:
pd_meta = pd.read_csv(f"{data_dir}/processed/observations/{city}/{network}/{city}_{network}_metadata.csv")
pd_meta.rename(columns={'name':'code'}, inplace=True)
obs_attributes = classify_stations_from_ucdb_city(pd_meta, ucdb_city)
obs_attributes = obs_attributes[obs_attributes["code"].isin(stations)]

pd_meta = pd.read_csv(f"{data_dir}/processed/observations/{city}/{network}/{city}_{network}_metadata.csv")
pd_meta.rename(columns={'name':'code'}, inplace=True)
obs_attributes_lczbased = classify_stations_from_LCZ(pd_meta)
obs_attributes_lczbased = obs_attributes_lczbased[obs_attributes_lczbased["code"].isin(stations)]
obs_attributes_lczbased

In [ ]:
for station in obs_timeseries.columns:
    print(f"Variable: {station}, nan count: {obs_timeseries[station].isnull().sum().item()}")

### Load synoptic stations (national meteorological stations)

In [ ]:
network_m='meteo_stations'
subset_m= ''
start_m = start
stop_m = stop


# load station observation data (preproccessed: monthly mean of daily data)
obs_path_tasmin_m = f"{data_dir}/processed/observations/{city}/{network_m}/datasets/{subset_m}/{city}_{network_m}{subset_m}_monthlymean_daily{var[3:]}.nc"
ds_obs_tasmin_m = xr.open_dataset(obs_path_tasmin_m, engine="netcdf4", chunks="auto").sel(time=slice(start,stop))
obs_timeseries_m = ds_obs_tasmin_m.to_dataframe().copy()
#obs_timeseries.drop('UKE00105909', axis='columns')
stations_m = obs_timeseries_m.columns.to_list()
print(f"{len(stations_m)} available stations: {stations_m}")
pd_meta = pd.read_csv(f"{data_dir}/processed/observations/{city}/{network_m}/{city}_{network_m}_metadata.csv")
pd_meta.rename(columns={'name':'code'}, inplace=True)
obs_attributes_m = classify_stations_from_ucdb_city(pd_meta, ucdb_city)
obs_attributes_m = obs_attributes_m[obs_attributes_m["code"].isin(stations_m)]
obs_attributes_m

In [ ]:

for station in obs_timeseries_m.columns:
    print(f"station: {station}, nan count: {obs_timeseries_m[station].isnull().sum().item()}")


In [ ]:
latex_table = obs_attributes[["code", "LCZ", "lat", "lon", "inside_city"]].rename(columns={'code':'Station', 'LCZ':'LCZ class', 'lat':'Latitude', 'lon':'Longitude', 'inside_city':'Inside city (UCDB)'}).copy()
latex_table.to_latex(f"{save_dir}/obs_attributes_{city}_{network}_{subset}.tex", index=False )
latex_table_m = obs_attributes_m[["code", "LCZ", "lat", "lon", "inside_city"]].rename(columns={'code':'Station', 'LCZ':'LCZ class', 'lat':'Latitude', 'lon':'Longitude', 'inside_city':'Inside city (UCDB)'}).copy()
latex_table_m.to_latex(f"{save_dir}/obs_attributes_m_{city}_{network}_{subset}.tex", index=False)

## UHI Urclim analysis

### UHI Urclim : urban network , ubcd classification

Paper method: **a2a_UCN_uc** (code: `urclim_un_ucdb`).

In [ ]:
UHI = UrbanIsland(
    ds=ds_RCM_urb[var],
    urban_vicinity=urmask,
    anomaly="abs",  # or "rel"
    obs_attributes=obs_attributes,
    obs_timeseries=obs_timeseries
)
report = UHI.check_obs_sim_overlap()

#calculate the annual cycle of the UHI for thobservations
UHI.compute_annual_cycle()
ds_anomaly = UHI.ds_annual_cycle

rural_anomaly = ds_anomaly.where(UHI.urban_vicinity['urmask'] == 0)
urban_anomaly = ds_anomaly.where(UHI.urban_vicinity['urmask'] == 1)

y_dim, x_dim = UHI._get_spatial_dim_names(ds_anomaly)       

urban_mean = urban_anomaly.mean(dim = [y_dim, x_dim]).compute()
rural_mean = rural_anomaly.mean(dim = [y_dim, x_dim]).compute()

uhi_urclim_un_ucdb_sim = urban_mean.values
uhi_urclim_un_ucdb_obs = UHI.obs_annual_cycle['urban_mean'].values


fig = UHI.plot_UI_map(
    city_name = city,
    ucdb_city = ucdb_city, 
)
fig.savefig(f"{plot_dir}/anamolymap_UN_UCDBclass_{var}.pdf", bbox_inches='tight')

fig = UHI.plot_UI_annual_cycle(
    ds_anomaly = ds_anomaly,
    obs_anomaly = UHI.obs_annual_cycle,
    percentiles = [5,25],
    gridcell_series = True, 
    city_name = city,
    vmin = -1, vmax =3.5)
fig.savefig(f"{plot_dir}/annualcycle_UN_UCDBclass_{var}.pdf", bbox_inches='tight')

### UHI Urclim: urban network, LCZ classification

Paper method: **a2a_UCN_lcz** (code: `urclim_un_lcz`).

In [ ]:
UHI_lcz = UrbanIsland(
    ds=ds_RCM_urb[var],
    urban_vicinity=urmask,
    anomaly="abs",  # or "rel"
    obs_attributes=obs_attributes_lczbased,
    obs_timeseries=obs_timeseries
)
report = UHI_lcz.check_obs_sim_overlap()

#calculate the annual cycle of the UHI for thobservations
UHI_lcz.compute_annual_cycle()
ds_anomaly = UHI_lcz.ds_annual_cycle

rural_anomaly = ds_anomaly.where(UHI_lcz.urban_vicinity['urmask'] == 0)
urban_anomaly = ds_anomaly.where(UHI_lcz.urban_vicinity['urmask'] == 1)

y_dim, x_dim = UHI_lcz._get_spatial_dim_names(ds_anomaly)       

urban_mean = urban_anomaly.mean(dim = [y_dim, x_dim]).compute()
rural_mean = rural_anomaly.mean(dim = [y_dim, x_dim]).compute()

uhi_urclim_un_lcz_sim = urban_mean.values
uhi_urclim_un_lcz_obs = UHI_lcz.obs_annual_cycle['urban_mean'].values


fig = UHI_lcz.plot_UI_map(
    city_name = city,
    ucdb_city = ucdb_city, 
)
fig.savefig(f"{plot_dir}/anamolymap_UN_LCZclass_{var}", bbox_inches='tight')

fig = UHI_lcz.plot_UI_annual_cycle(
    ds_anomaly = ds_anomaly,
    obs_anomaly = UHI_lcz.obs_annual_cycle,
    percentiles = [5,25],
    gridcell_series = True, 
    city_name = city,
    vmin = -1, vmax =3.5)
fig.savefig(f"{plot_dir}/annualcycle_UN_LCZclass_{var}.pdf", bbox_inches='tight')

### UHI urclim, national meteorological stations, UBCD classificatino

Paper method: **a2a_MS_uc** (code: `urclim_ms_ucdb`).

In [ ]:
UHI_m = UrbanIsland(
    ds=ds_RCM_urb[var],
    urban_vicinity=urmask,
    anomaly="abs",  # or "rel"
    obs_attributes=obs_attributes_m,
    obs_timeseries=obs_timeseries_m
)
report = UHI_m.check_obs_sim_overlap()

#calculate the annual cycle of the UHI for thobservations
UHI_m.compute_annual_cycle()
ds_anomaly = UHI_m.ds_annual_cycle

rural_anomaly = ds_anomaly.where(UHI_m.urban_vicinity['urmask'] == 0)
urban_anomaly = ds_anomaly.where(UHI_m.urban_vicinity['urmask'] == 1)

y_dim, x_dim = UHI_m._get_spatial_dim_names(ds_anomaly)       

urban_mean = urban_anomaly.mean(dim = [y_dim, x_dim]).compute()
rural_mean = rural_anomaly.mean(dim = [y_dim, x_dim]).compute()

uhi_urclim_ms_sim = urban_mean.values
uhi_urclim_ms_obs = UHI_m.obs_annual_cycle['urban_mean'].values


fig = UHI_m.plot_UI_map(
    city_name = city,
    ucdb_city = ucdb_city, 
)
fig.savefig(f"{plot_dir}/anamolymap_MS_UCDBclass_{var}", bbox_inches='tight')

fig = UHI_m.plot_UI_annual_cycle(
    ds_anomaly = ds_anomaly,
    obs_anomaly = UHI_m.obs_annual_cycle,
    percentiles = [5,25],
    gridcell_series = True, 
    city_name = city,
    vmin = -1, vmax =3.5)

fig.savefig(f"{plot_dir}/annualcycle_MS_UCDBclass_{var}.pdf", bbox_inches='tight')

# Point to point analysis 

In [ ]:
proj_dict = ds_RCM_urb.crs.attrs
plot_proj = ccrs.LambertConformal(central_latitude=proj_dict["latitude_of_projection_origin"], central_longitude=proj_dict["longitude_of_central_meridian"], standard_parallels=proj_dict["standard_parallel"], globe=ccrs.Globe(semimajor_axis=proj_dict["earth_radius"], semiminor_axis=proj_dict["earth_radius"]))
plot_proj

Paper method **p2p_UCN_uc**: each station is compared with its own grid cell; UHIN = mean(urban) - mean(rural), stations classified with the UCDB urban centre.

In [ ]:
fig,ax = plt.subplots(figsize=(10, 6))
uhi_p2p_un_core= compute_UHI_p2p( obs_attributes, obs_timeseries, ds_RCM_urb, proj=plot_proj,mean=False, variable=var)
uhi_p2p_un_ucdb_obs, uhi_p2p_un_ucdb_sim = plot_UHI_p2p_cycle( ax, uhi_p2p_un_core, mean=False,  percentiles=[20], plot_individual=False)
plt.show()

Paper method **p2p_UCN_lcz**: as above, stations classified with LCZ (urban: LCZ 1-10, rural: LCZ A-G).

In [ ]:
fig,ax = plt.subplots(figsize=(10, 6))
uhi_p2p_un_lcz = compute_UHI_p2p( obs_attributes_lczbased, obs_timeseries, ds_RCM_urb, proj=plot_proj,mean=False, variable=var)
uhi_p2p_un_lcz_obs, uhi_p2p_un_lcz_sim = plot_UHI_p2p_cycle( ax, uhi_p2p_un_lcz, mean=False,  percentiles=[20], plot_individual=False)
plt.show()

Paper method **p2p_MS_uc**: as above, with the meteorological service stations.

In [ ]:
uhi_un_p2p_fig,ax = plt.subplots(figsize=(10, 6))
uhi_p2p_ms = compute_UHI_p2p( obs_attributes_m, obs_timeseries_m, ds_RCM_urb, proj=plot_proj,mean=False, variable=var)
uhi_p2p_ms_obs, uhi_p2p_ms_sim = plot_UHI_p2p_cycle( ax, uhi_p2p_ms, mean=False,  percentiles=[20], plot_individual=False)
plt.show()

# Combine and save P2P, URCLIM and Blended data in dataset and dataframe

Adds the p2a result (`blend`) from the blended-approach notebook and saves the seven methods (sim, obs, bias per month) to `results/data/<City>/`.

In [ ]:
sim_ts = xr.open_dataset(f"{save_dir}/method_2_blended_approach\{city}_UHI_method2_simulation.nc", engine="netcdf4")
obs_ts = xr.open_dataset(f"{save_dir}/method_2_blended_approach\{city}_UHI_method2_observations.nc", engine="netcdf4")

uhi_blend_obs = obs_ts[f'UHI_{var}'].groupby('time.month').mean().values
uhi_blend_sim = sim_ts[f'UHI_{var}'].groupby('time.month').mean().values

In [ ]:
if var=='tasmin':
    suffix = 'N'
elif var=='tasmax':
    suffix = 'X'

#store all biases in a dataset 
uhis = xr.Dataset(
    {   f'UHI{suffix}_p2p_un_ucdb_obs': (["month"], uhi_p2p_un_ucdb_obs),
        f'UHI{suffix}_p2p_un_ucdb_sim': (["month"], uhi_p2p_un_ucdb_sim),
        f'UHI{suffix}_p2p_un_ucdb_bias': (["month"], uhi_p2p_un_ucdb_sim- uhi_p2p_un_ucdb_obs),

        f'UHI{suffix}_p2p_un_lcz_obs': (["month"], uhi_p2p_un_lcz_obs),
        f'UHI{suffix}_p2p_un_lcz_sim': (["month"], uhi_p2p_un_lcz_sim),
        f'UHI{suffix}_p2p_un_lcz_bias': (["month"], uhi_p2p_un_lcz_sim- uhi_p2p_un_lcz_obs),


        f'UHI{suffix}_p2p_ms_ucdb_obs': (["month"], uhi_p2p_ms_obs),
        f'UHI{suffix}_p2p_ms_ucdb_sim': (["month"], uhi_p2p_ms_sim),
         f'UHI{suffix}_p2p_ms_ucdb_bias': (["month"], uhi_p2p_ms_sim- uhi_p2p_ms_obs),


        #-----------------------------------------------------------
        f'UHI{suffix}_urclim_un_ucdb_obs': (["month"], uhi_urclim_un_ucdb_obs),
        f'UHI{suffix}_urclim_un_ucdb_sim': (["month"], uhi_urclim_un_ucdb_sim),
        f'UHI{suffix}_urclim_un_ucdb_bias': (["month"], uhi_urclim_un_ucdb_sim- uhi_urclim_un_ucdb_obs),

        f'UHI{suffix}_urclim_un_lcz_obs': (["month"], uhi_urclim_un_lcz_obs),
        f'UHI{suffix}_urclim_un_lcz_sim': (["month"], uhi_urclim_un_lcz_sim),
        f'UHI{suffix}_urclim_un_lcz_bias': (["month"], uhi_urclim_un_lcz_sim- uhi_urclim_un_lcz_obs),


        f'UHI{suffix}_urclim_ms_ucdb_obs': (["month"], uhi_urclim_ms_obs),
        f'UHI{suffix}_urclim_ms_ucdb_sim': (["month"], uhi_urclim_ms_sim),
        f'UHI{suffix}_urclim_ms_ucdb_bias': (["month"], uhi_urclim_ms_sim- uhi_urclim_ms_obs),

        #-----------------------------------------------------------
        f'UHI{suffix}_blend_ms_ucdb_obs': (["month"], uhi_blend_obs),
        f'UHI{suffix}_blend_ms_ucdb_sim': (["month"], uhi_blend_sim),
        f'UHI{suffix}_blend_ms_ucdb_bias': (["month"], uhi_blend_sim- uhi_blend_obs),

    },
    coords={
        "month": np.arange(1, 13)
    }
)



uhis.to_netcdf(f"{save_dir}/UHI_{city}_annualcycle_method_comparison_{var}.nc")
uhis_df = uhis.to_dataframe()
uhis_df.to_csv(f"{save_dir}/UHI_{city}_annualcycle_method_comparison_{var}.csv")
uhis_df

## Diagnostic plots
Annual cycles of simulated, observed UHIN and bias for the seven methods (not paper figures; Fig. 2 is made in `scripts/plotting/fig2_bias_spread_rennes.ipynb`).

In [ ]:

fig, ax = plt.subplots(figsize=(15, 7))


ax.plot(uhis[f'UHI{suffix}_urclim_un_ucdb_obs'], label=f'UHI{suffix}_urclim_un_ucdb_obs', color='darkred', linestyle='--', marker='o', lw=4)
ax.plot(uhis[f'UHI{suffix}_urclim_un_lcz_obs'], label=f'UHI{suffix}_urclim_un_lcz_obs', color='peru', linestyle='--', marker='o', lw=4)
ax.plot(uhis[f'UHI{suffix}_urclim_ms_ucdb_obs'], label=f'UHI{suffix}_urclim_ms_ucdb_obs', color='red', linestyle='--', marker='o', lw=4)


ax.plot(uhis[f'UHI{suffix}_urclim_un_ucdb_sim'], color='darkred', linestyle=(0,(5,10)), lw=4)
ax.plot(uhis[f'UHI{suffix}_urclim_un_lcz_sim'], color='peru', linestyle=(5,(5,10)), lw=4)
ax.plot(uhis[f'UHI{suffix}_urclim_ms_ucdb_sim'], color='red', linestyle=(10,(5,10)), lw=4)


ax.plot(uhis[f'UHI{suffix}_p2p_un_ucdb_sim'], label=f'UHI{suffix}_p2p_un_ucdb_sim', color='darkred', lw=4)
ax.plot(uhis[f'UHI{suffix}_p2p_un_lcz_sim'], label=f'UHI{suffix}_p2p_un_lcz_sim', color='peru', lw=4)
ax.plot(uhis[f'UHI{suffix}_p2p_ms_ucdb_sim'], label=f'UHI{suffix}_p2p_ms_ucdb_sim', color='red', lw=4)

ax.plot(uhis[f'UHI{suffix}_blend_ms_ucdb_sim'], label=f'UHI{suffix}_blend_ms_ucdb_sim', color='salmon', lw=4)
ax.plot(uhis[f'UHI{suffix}_blend_ms_ucdb_obs'], label=f'UHI{suffix}_blend_ms_ucdb_obs', color='salmon', linestyle='--', marker='o', lw=4)


ax.plot(np.zeros(12), color='green', lw=3,marker='o', label= 'rural_ref')

ax.set_title(f"{city} UHI_{var} \n all methods ", fontsize=16)
ax.set_xlabel("Month", fontsize=14)
#set x ticks to show month names
ax.set_xticks(np.arange(0, 12))
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'], fontsize=12)

ax.set_ylabel(f"UHI_{var} [°C]", fontsize=14)
ax.grid()
plt.legend()

fig.show()

In [ ]:

fig, ax = plt.subplots(figsize=(15, 7))


ax.plot(uhis[f'UHI{suffix}_urclim_un_ucdb_obs'], label=f'UHI{suffix}_urclim_un_ucdb_obs', color='darkred', linestyle='--', marker='o', lw=4, alpha = 0.2)
ax.plot(uhis[f'UHI{suffix}_urclim_un_lcz_obs'], label=f'UHI{suffix}_urclim_un_lcz_obs', color='peru', linestyle='--', marker='o', lw=4, alpha = 0.2)
ax.plot(uhis[f'UHI{suffix}_urclim_ms_ucdb_obs'], label=f'UHI{suffix}_urclim_ms_ucdb_obs', color='red', linestyle='--', marker='o', lw=4)


ax.plot(uhis[f'UHI{suffix}_urclim_un_ucdb_sim'], color='darkred', linestyle=(0,(5,10)), lw=4)
ax.plot(uhis[f'UHI{suffix}_urclim_un_lcz_sim'], color='peru', linestyle=(5,(5,10)), lw=4)
ax.plot(uhis[f'UHI{suffix}_urclim_ms_ucdb_sim'], color='red', linestyle=(10,(5,10)), lw=4)


ax.plot(uhis[f'UHI{suffix}_p2p_un_ucdb_sim'], label=f'UHI{suffix}_p2p_un_ucdb_sim', color='darkred', lw=4, alpha = 0.2)
ax.plot(uhis[f'UHI{suffix}_p2p_un_lcz_sim'], label=f'UHI{suffix}_p2p_un_lcz_sim', color='peru', lw=4, alpha = 0.2)
ax.plot(uhis[f'UHI{suffix}_p2p_ms_ucdb_sim'], label=f'UHI{suffix}_p2p_ms_ucdb_sim', color='red', lw=4)

ax.plot(uhis[f'UHI{suffix}_blend_ms_ucdb_sim'], label=f'UHI{suffix}_blend_ms_ucdb_sim', color='salmon', lw=4)
ax.plot(uhis[f'UHI{suffix}_blend_ms_ucdb_obs'], label=f'UHI{suffix}_blend_ms_ucdb_obs', color='salmon', linestyle='--', marker='o', lw=4)


ax.plot(np.zeros(12), color='green', lw=3,marker='o', label= 'rural_ref')

ax.set_title(f"{city} UHI_{var}  \n Meteo station network comparison : P2P vs URCLIM vs BLEND ", fontsize=16)
ax.set_xlabel("Month", fontsize=14)
#set x ticks to show month names
ax.set_xticks(np.arange(0, 12))
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'], fontsize=12)

ax.set_ylabel(f"UHI_{var} [°C]", fontsize=14)
ax.grid()
plt.legend()

fig.show()

In [ ]:

fig, ax = plt.subplots(figsize=(15, 7))


ax.plot(uhis[f'UHI{suffix}_urclim_un_ucdb_obs'], label=f'UHI{suffix}_urclim_un_ucdb_obs', color='darkred', linestyle='--', marker='o', lw=4)
ax.plot(uhis[f'UHI{suffix}_urclim_un_lcz_obs'], label=f'UHI{suffix}_urclim_un_lcz_obs', color='peru', linestyle='--', marker='o', lw=4)
ax.plot(uhis[f'UHI{suffix}_urclim_ms_ucdb_obs'], label=f'UHI{suffix}_urclim_ms_ucdb_obs', color='red', linestyle='--', marker='o', lw=4, alpha = 0.2)


ax.plot(uhis[f'UHI{suffix}_urclim_un_ucdb_sim'], color='darkred', linestyle=(0,(5,10)), lw=4)
ax.plot(uhis[f'UHI{suffix}_urclim_un_lcz_sim'], color='peru', linestyle=(5,(5,10)), lw=4)
ax.plot(uhis[f'UHI{suffix}_urclim_ms_ucdb_sim'], color='red', linestyle=(10,(5,10)), lw=4)


ax.plot(uhis[f'UHI{suffix}_p2p_un_ucdb_sim'], label=f'UHI{suffix}_p2p_un_ucdb_sim', color='darkred', lw=4)
ax.plot(uhis[f'UHI{suffix}_p2p_un_lcz_sim'], label=f'UHI{suffix}_p2p_un_lcz_sim', color='peru', lw=4)
ax.plot(uhis[f'UHI{suffix}_p2p_ms_ucdb_sim'], label=f'UHI{suffix}_p2p_ms_ucdb_sim', color='red', lw=4, alpha = 0.2)

ax.plot(uhis[f'UHI{suffix}_blend_ms_ucdb_sim'], label=f'UHI{suffix}_blend_ms_ucdb_sim', color='salmon', lw=4, alpha = 0.2)
ax.plot(uhis[f'UHI{suffix}_blend_ms_ucdb_obs'], label=f'UHI{suffix}_blend_ms_ucdb_obs', color='salmon', linestyle='--', marker='o', lw=4, alpha = 0.2)


ax.plot(np.zeros(12), color='green', lw=3,marker='o', label= 'rural_ref')

ax.set_title(f"{city} UHI_{var} \n Urban network comparison : P2P vs URCLIM ", fontsize=16)
ax.set_xlabel("Month", fontsize=14)
#set x ticks to show month names
ax.set_xticks(np.arange(0, 12))
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'], fontsize=12)

ax.set_ylabel(f"UHI_{var} [°C]", fontsize=14)
ax.grid()
plt.legend()

fig.show()

In [ ]:

fig, ax = plt.subplots(figsize=(15, 7))


ax.plot(uhis[f'UHI{suffix}_urclim_un_ucdb_obs'], label=f'UHI{suffix}_urclim_un_ucdb_obs', color='darkred', linestyle='--', marker='o', lw=4)
ax.plot(uhis[f'UHI{suffix}_urclim_un_lcz_obs'], label=f'UHI{suffix}_urclim_un_lcz_obs', color='peru', linestyle='--', marker='o', lw=4)
ax.plot(uhis[f'UHI{suffix}_urclim_ms_ucdb_obs'], label=f'UHI{suffix}_urclim_ms_ucdb_obs', color='red', linestyle='--', marker='o', lw=4)


ax.plot(uhis[f'UHI{suffix}_urclim_un_ucdb_sim'], color='darkred', linestyle=(0,(5,10)), lw=4, alpha = 0.2)
ax.plot(uhis[f'UHI{suffix}_urclim_un_lcz_sim'], color='peru', linestyle=(5,(5,10)), lw=4, alpha = 0.2)
ax.plot(uhis[f'UHI{suffix}_urclim_ms_ucdb_sim'], color='red', linestyle=(10,(5,10)), lw=4, alpha = 0.2)


ax.plot(uhis[f'UHI{suffix}_p2p_un_ucdb_sim'], label=f'UHI{suffix}_p2p_un_ucdb_sim', color='darkred', lw=4)
ax.plot(uhis[f'UHI{suffix}_p2p_un_lcz_sim'], label=f'UHI{suffix}_p2p_un_lcz_sim', color='peru', lw=4)
ax.plot(uhis[f'UHI{suffix}_p2p_ms_ucdb_sim'], label=f'UHI{suffix}_p2p_ms_ucdb_sim', color='red', lw=4)

ax.plot(uhis[f'UHI{suffix}_blend_ms_ucdb_sim'], label=f'UHI{suffix}_blend_ms_ucdb_sim', color='salmon', lw=4, alpha = 0.2)
ax.plot(uhis[f'UHI{suffix}_blend_ms_ucdb_obs'], label=f'UHI{suffix}_blend_ms_ucdb_obs', color='salmon', linestyle='--', marker='o', lw=4, alpha = 0.2)


ax.plot(np.zeros(12), color='green', lw=3,marker='o', label= 'rural_ref')

ax.set_title(f"{city} UHI_{var} \n P2P comparison : UN_UCDB, UN_LCZ, MS_UCDB ", fontsize=16)
ax.set_xlabel("Month", fontsize=14)
#set x ticks to show month names
ax.set_xticks(np.arange(0, 12))
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'], fontsize=12)

ax.set_ylabel(f"UHI_{var} [°C]", fontsize=14)
ax.grid()
plt.legend()

fig.show()

In [ ]:

fig, ax = plt.subplots(figsize=(15, 7))


ax.plot(uhis[f'UHI{suffix}_urclim_un_ucdb_obs'], label=f'UHI{suffix}_urclim_un_ucdb_obs', color='darkred', linestyle='--', marker='o', lw=4)
ax.plot(uhis[f'UHI{suffix}_urclim_un_lcz_obs'], label=f'UHI{suffix}_urclim_un_lcz_obs', color='peru', linestyle='--', marker='o', lw=4)
ax.plot(uhis[f'UHI{suffix}_urclim_ms_ucdb_obs'], label=f'UHI{suffix}_urclim_ms_ucdb_obs', color='red', linestyle='--', marker='o', lw=4)


ax.plot(uhis[f'UHI{suffix}_urclim_un_ucdb_sim'], color='darkred', linestyle=(0,(5,10)), lw=4)
ax.plot(uhis[f'UHI{suffix}_urclim_un_lcz_sim'], color='peru', linestyle=(5,(5,10)), lw=4)
ax.plot(uhis[f'UHI{suffix}_urclim_ms_ucdb_sim'], color='red', linestyle=(10,(5,10)), lw=4)


ax.plot(uhis[f'UHI{suffix}_p2p_un_ucdb_sim'], label=f'UHI{suffix}_p2p_un_ucdb_sim', color='darkred', lw=4, alpha = 0.2)
ax.plot(uhis[f'UHI{suffix}_p2p_un_lcz_sim'], label=f'UHI{suffix}_p2p_un_lcz_sim', color='peru', lw=4, alpha = 0.2)
ax.plot(uhis[f'UHI{suffix}_p2p_ms_ucdb_sim'], label=f'UHI{suffix}_p2p_ms_ucdb_sim', color='red', lw=4, alpha = 0.2)

ax.plot(uhis[f'UHI{suffix}_blend_ms_ucdb_sim'], label=f'UHI{suffix}_blend_ms_ucdb_sim', color='salmon', lw=4, alpha = 0.2)
ax.plot(uhis[f'UHI{suffix}_blend_ms_ucdb_obs'], label=f'UHI{suffix}_blend_ms_ucdb_obs', color='salmon', linestyle='--', marker='o', lw=4, alpha = 0.2)


ax.plot(np.zeros(12), color='green', lw=3,marker='o', label= 'rural_ref')

ax.set_title(f" {city} UHI_{var} \n URCLIM comparison : UN_UCDB, UN_LCZ, MS_UCDB ", fontsize=16)
ax.set_xlabel("Month", fontsize=14)
#set x ticks to show month names
ax.set_xticks(np.arange(0, 12))
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'], fontsize=12)

ax.set_ylabel(f"UHI_{var} [°C]", fontsize=14)
ax.grid()
plt.legend()


In [ ]:


fig = plt.figure(figsize=(18,12))
gs = fig.add_gridspec(2, 2)

ax1 = fig.add_subplot(gs[0, 0])   # top left
ax2 = fig.add_subplot(gs[0, 1])   # top right
ax3 = fig.add_subplot(gs[1, :])   # bottom spans both columns

pos = ax3.get_position()

ax3.set_position([
    pos.x0 + 0.05 ,   # move right
    pos.y0,
    pos.width - 0.20,  # make narrower
    pos.height
])
#plot simulation UHI 
ax1.plot(uhis[f'UHI{suffix}_urclim_un_ucdb_obs'], label='urclim_un_ucdb_obs', color='red',  linestyle=(0,(5,7)), marker='o', lw=4, alpha=0.2)
ax1.plot(uhis[f'UHI{suffix}_urclim_un_lcz_obs'], label='urclim_un_lcz_obs', color='firebrick', linestyle=(0,(5,7)), marker='o', lw=4, alpha=0.2)
ax1.plot(uhis[f'UHI{suffix}_urclim_ms_ucdb_obs'], label='urclim_ms_ucdb_obs', color='crimson', linestyle=(0,(5,7)), marker='o', lw=4, alpha=0.2)


ax1.plot(uhis[f'UHI{suffix}_p2p_un_ucdb_obs'], label='urclim_un_ucdb_obs', color='orangered', linestyle=(6,(5,7)), marker='o', lw=4, alpha=0.2)
ax1.plot(uhis[f'UHI{suffix}_p2p_un_lcz_obs'], label='urclim_un_lcz_obs', color='darkorange', linestyle=(6,(5,7)), marker='o', lw=4, alpha=0.2)
ax1.plot(uhis[f'UHI{suffix}_p2p_ms_ucdb_obs'], label='urclim_ms_ucdb_obs', color='salmon', linestyle=(6,(5,7)), marker='o', lw=4, alpha=0.2)

ax1.plot(uhis[f'UHI{suffix}_urclim_un_ucdb_sim'],label = 'urclim_un_ucdb_sim' , color='red', linestyle=(0,(5,10)), lw=4, zorder=10)
ax1.plot(uhis[f'UHI{suffix}_urclim_un_lcz_sim'],label = 'urclim_un_lcz_sim', color='firebrick', linestyle=(5,(5,10)), lw=4,zorder=10)
ax1.plot(uhis[f'UHI{suffix}_urclim_ms_ucdb_sim'],label = 'urclim_ms_ucdb_sim', color='crimson', linestyle=(10,(5,10)), lw=4,zorder=10)


ax1.plot(uhis[f'UHI{suffix}_p2p_un_ucdb_sim'], label='p2p_un_ucdb_sim', color='orangered', lw=4)
ax1.plot(uhis[f'UHI{suffix}_p2p_un_lcz_sim'], label='p2p_un_lcz_sim', color='darkorange', lw=4)
ax1.plot(uhis[f'UHI{suffix}_p2p_ms_ucdb_sim'], label='p2p_ms_ucdb_sim', color='salmon', lw=4)

ax1.plot(uhis[f'UHI{suffix}_blend_ms_ucdb_sim'], label='blend_ms_ucdb_sim', color='darkred', lw=4)
ax1.plot(uhis[f'UHI{suffix}_blend_ms_ucdb_obs'], label='blend_ms_ucdb_obs', color='darkred', linestyle='--', marker='o', lw=4,  alpha=0.2)


ax1.plot(np.zeros(12), color='green', lw=3,marker='o', label= 'rural_ref')


ax1.set_title(f"simulated UHI{suffix}", fontsize=16)
ax1.set_xticks(np.arange(0, 12))
ax1.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'], fontsize=12)

ax1.set_ylabel(f"UHI{suffix} [°C]", fontsize=14)
ax1.grid()

#plot observations UHI

ax2.plot(uhis[f'UHI{suffix}_urclim_un_ucdb_obs'], label='urclim_un_ucdb_obs', color='red',  linestyle=(0,(5,7)), marker='o', lw=4)
ax2.plot(uhis[f'UHI{suffix}_urclim_un_lcz_obs'], label='urclim_un_lcz_obs', color='firebrick', linestyle=(0,(5,7)), marker='o', lw=4)
ax2.plot(uhis[f'UHI{suffix}_urclim_ms_ucdb_obs'], label='urclim_ms_ucdb_obs', color='crimson', linestyle=(0,(5,7)), marker='o', lw=4)


ax2.plot(uhis[f'UHI{suffix}_p2p_un_ucdb_obs'], label='urclim_un_ucdb_obs', color='orangered', linestyle=(6,(5,7)), marker='o', lw=4)
ax2.plot(uhis[f'UHI{suffix}_p2p_un_lcz_obs'], label='urclim_un_lcz_obs', color='darkorange', linestyle=(6,(5,7)), marker='o', lw=4)
ax2.plot(uhis[f'UHI{suffix}_p2p_ms_ucdb_obs'], label='urclim_ms_ucdb_obs', color='salmon', linestyle=(6,(5,7)), marker='o', lw=4)

ax2.plot(uhis[f'UHI{suffix}_urclim_un_ucdb_sim'],label = 'urclim_un_ucdb_sim' , color='red', linestyle=(0,(5,10)), lw=4, alpha=0.2)
ax2.plot(uhis[f'UHI{suffix}_urclim_un_lcz_sim'],label = 'urclim_un_lcz_sim', color='firebrick', linestyle=(5,(5,10)), lw=4, alpha=0.2)
ax2.plot(uhis[f'UHI{suffix}_urclim_ms_ucdb_sim'],label = 'urclim_ms_ucdb_sim', color='crimson', linestyle=(10,(5,10)), lw=4, alpha=0.2)


ax2.plot(uhis[f'UHI{suffix}_p2p_un_ucdb_sim'], label='p2p_un_ucdb_sim', color='orangered', lw=4, alpha=0.2)
ax2.plot(uhis[f'UHI{suffix}_p2p_un_lcz_sim'], label='p2p_un_lcz_sim', color='darkorange', lw=4, alpha=0.2)
ax2.plot(uhis[f'UHI{suffix}_p2p_ms_ucdb_sim'], label='p2p_ms_ucdb_sim', color='salmon', lw=4, alpha=0.2)

ax2.plot(uhis[f'UHI{suffix}_blend_ms_ucdb_sim'], label='blend_ms_ucdb_sim', color='darkred', lw=4, alpha=0.2)
ax2.plot(uhis[f'UHI{suffix}_blend_ms_ucdb_obs'], label='blend_ms_ucdb_obs', color='darkred', linestyle='--', marker='o', lw=4)


ax2.plot(np.zeros(12), color='green', lw=3,marker='o', label= 'rural_ref')


ax2.set_title(f"observed UHI{suffix}", fontsize=16)
#set x ticks to show month names
ax2.set_xticks(np.arange(0, 12))
ax2.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'], fontsize=12)
ax2.grid()

#plot biases



ax3.plot(uhis[f'UHI{suffix}_urclim_un_ucdb_bias'], label=f'UHI{suffix}_urclim_un_ucdb_bias', color='red',lw=4,  ls = (0,(3,1,1,1)) )
ax3.plot(uhis[f'UHI{suffix}_urclim_un_lcz_bias'], label=f'UHI{suffix}_urclim_un_lcz_bias', color='firebrick', lw=4 ,ls = (0,(3,1,1,1)) )
ax3.plot(uhis[f'UHI{suffix}_urclim_ms_ucdb_bias'], label=f'UHI{suffix}_urclim_ms_ucdb_bias', color='crimson', lw=4,ls = (0,(3,1,1,1)) )
ax3.plot(uhis[f'UHI{suffix}_p2p_un_ucdb_bias'], label=f'UHI{suffix}_p2p_un_ucdb_bias', color='orangered', lw=4,ls = (0,(3,1,1,1)) )
ax3.plot(uhis[f'UHI{suffix}_p2p_un_lcz_bias'], label=f'UHI{suffix}_p2p_un_lcz_bias', color='darkorange', lw=4,ls = (0,(3,1,1,1)) )
ax3.plot(uhis[f'UHI{suffix}_p2p_ms_ucdb_bias'], label=f'UHI{suffix}_p2p_ms_ucdb_bias', color='salmon', lw=4,ls = (0,(3,1,1,1)) )
ax3.plot(uhis[f'UHI{suffix}_blend_ms_ucdb_bias'], label=f'UHI{suffix}_blend_ms_ucdb_bias', color='darkred', lw=4,ls = (0,(3,1,1,1)) )


#plot horizontal line at y=0
ax3.axhline(y=0, color='black', lw=2, ls='dotted')
#plot a vertical line at x = 9
ax3.axvline(x=8, color='black', lw=1.3, )

#plot the value of p2p_MS_ucdb_bias at x=8 next to the vertical line
p2p_ms_ucdb_bias_value = uhis[f'UHI{suffix}_p2p_ms_ucdb_bias'].values[8]
ax3.text(8.2, -0.8, f"{p2p_ms_ucdb_bias_value:.2f}", color='salmon', fontsize=12, va='bottom')

#plot the value of urclim_un_ucdb_bias at x=8 next to the vertical line
urclim_un_ucdb_bias_value = uhis[f'UHI{suffix}_urclim_un_ucdb_bias'].values[8]
ax3.text(8.1, -0.225, f"{urclim_un_ucdb_bias_value:.2f}", color='red', fontsize=12, va='bottom')

#plot the value of blend_ms_ucdb_bias at x=8 next to the vertical line
blend_ms_ucdb_bias_value = uhis[f'UHI{suffix}_blend_ms_ucdb_bias'].values[8]
ax3.text(8.1, 1.86, f"{blend_ms_ucdb_bias_value:.2f}", color='darkred', fontsize=12, va='bottom')

ax3.set_title(f" UHI{suffix} bias", fontsize=16)

ax3.set_xlabel("Month", fontsize=14)
#set x ticks to show month names
ax3.set_xticks(np.arange(0, 12))
ax3.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'], fontsize=12)

ax3.set_ylabel(f"UHI{suffix} bias [°C]", fontsize=14)
ax3.grid()

#plot legend 

# Hardcoded method legend
method_legend = [

    Line2D([0], [0], color='darkred', lw=4, label='p2a_MS_dist'),

    Line2D([0], [0], color='firebrick', lw=4, label='a2a_UCN_lcz'),
    Line2D([0], [0], color='crimson', lw=4, label='a2a_MS_uc'),
    Line2D([0], [0], color='red', lw=4, label='a2a_UCN_uc'),

    Line2D([0], [0], color='orangered', lw=4, label='p2p_UCN_uc'),
    Line2D([0], [0], color='darkorange', lw=4, label='p2p_UCN_lcz'),
    Line2D([0], [0], color='salmon', lw=4, label='p2p_MS_uc'),
    Line2D([0], [0], color='green', lw=4, label='Rural ref'),

]

# Hardcoded line style legend
type_legend = [
    Line2D([0], [0], color='black', lw=4, linestyle='-', label='Simulation'),
    Line2D([0], [0], color='black', lw=4, linestyle='--',marker='o', label='Observation'),
    Line2D([0], [0], color='black', lw=4, ls = (0,(3,1,1,1)), label='Bias'),

    
]

legend1 = ax3.legend(
    handles=method_legend,
    title="Evaluation method",
    loc="center left",
    bbox_to_anchor=(1.02,0.6),  
    borderaxespad=0.,
    fontsize=14
)
ax3.add_artist(legend1)

ax3.legend(
    handles=type_legend,
    title="Type",
    loc="upper left",
    bbox_to_anchor=(1.02, .25),
    borderaxespad=0., 
    fontsize=14
)




In [ ]:
# Fig. 2, variant D (Steven #42/#43): same layout as the cell above, with
# - one line width (4) for every line
# - simulated: short dashes, no markers; the 3 identical a2a curves cycle their 3 colors
# - observed: long dashes with markers; a2a and p2p share the curve -> 2 colors interleaved
# - bias: dash-dot, as before

col = {'p2a_MS_dist':'#4a0010', 'a2a_UCN_lcz':'#a50f15', 'a2a_MS_uc':'#ef3b2c', 'a2a_UCN_uc':'#c51b7d',
       'p2p_UCN_uc':'#ff7f00', 'p2p_UCN_lcz':'#ffd23f', 'p2p_MS_uc':'#fcae91'}
key = {'p2p_UCN_uc':'p2p_un_ucdb', 'p2p_UCN_lcz':'p2p_un_lcz', 'p2p_MS_uc':'p2p_ms_ucdb',
       'a2a_UCN_uc':'urclim_un_ucdb', 'a2a_UCN_lcz':'urclim_un_lcz', 'a2a_MS_uc':'urclim_ms_ucdb',
       'p2a_MS_dist':'blend_ms_ucdb'}
val = lambda m, k: uhis[f'UHI{suffix}_{key[m]}_{k}'].values
a2a = ['a2a_UCN_uc', 'a2a_UCN_lcz', 'a2a_MS_uc']
p2p = ['p2p_UCN_uc', 'p2p_UCN_lcz', 'p2p_MS_uc']
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
SIM = dict(lw=4, solid_capstyle='butt', dash_capstyle='butt')
OBS = dict(lw=4, marker='o', ms=7, mec='white', mew=0.8, dash_capstyle='butt')
BIAS = dict(lw=4, ls=(0,(3,1,1,1)))

fig = plt.figure(figsize=(18,12))
gs = fig.add_gridspec(2, 2)
ax1 = fig.add_subplot(gs[0, 0])   # top left
ax2 = fig.add_subplot(gs[0, 1], sharey=ax1)   # top right
ax3 = fig.add_subplot(gs[1, :])   # bottom spans both columns
pos = ax3.get_position()
ax3.set_position([pos.x0 + 0.05, pos.y0, pos.width - 0.20, pos.height])

# simulated (ax1) and observed (ax2) UHI; the other type is faded
for ax, main in ((ax1, 'sim'), (ax2, 'obs')):
    al = lambda k: 1.0 if k == main else 0.2
    for ma, mp in zip(a2a, p2p):
        ax.plot(val(ma, 'obs'), color=col[ma], ls=(0,(8,12)), alpha=al('obs'), **OBS)
        ax.plot(val(mp, 'obs'), color=col[mp], ls=(10,(8,12)), alpha=al('obs'), **OBS)
    ax.plot(val('p2a_MS_dist', 'obs'), color=col['p2a_MS_dist'], ls=(0,(8,2)), alpha=al('obs'), **OBS)
    for mp in p2p:
        ax.plot(val(mp, 'sim'), color=col[mp], ls=(0,(3,1)), alpha=al('sim'), **SIM)
    ax.plot(val('p2a_MS_dist', 'sim'), color=col['p2a_MS_dist'], ls=(0,(3,1)), alpha=al('sim'), **SIM)
    for i, ma in enumerate(a2a):
        ax.plot(val(ma, 'sim'), color=col[ma], ls=(4*i,(3,9)), alpha=al('sim'), zorder=10, **SIM)
    ax.plot(np.zeros(12), color='green', lw=4, marker='o')
    ax.set_xticks(np.arange(0, 12))
    ax.set_xticklabels(months, fontsize=12)
    ax.grid()
ax1.set_title(f"simulated UHI{suffix}", fontsize=16)
ax2.set_title(f"observed UHI{suffix}", fontsize=16)
ax1.set_ylabel(f"UHI{suffix} [°C]", fontsize=14)

# biases
for m, c in col.items():
    ax3.plot(val(m, 'bias'), color=c, **BIAS)
ax3.axhline(y=0, color='green', lw=4, marker='o')
ax3.axvline(x=8, color='black', lw=1.3)
# September bias values next to the vertical line
for m, xo, yo in (('p2p_MS_uc', 8.2, -0.8), ('a2a_UCN_uc', 8.1, -0.225), ('p2a_MS_dist', 8.1, 1.86)):
    ax3.text(xo, yo, f"{val(m, 'bias')[8]:.2f}", color=col[m], fontsize=12, va='bottom')
ax3.set_title(f" UHI{suffix} bias", fontsize=16)
ax3.set_xlabel("Month", fontsize=14)
ax3.set_xticks(np.arange(0, 12))
ax3.set_xticklabels(months, fontsize=12)
ax3.set_ylabel(f"UHI{suffix} bias [°C]", fontsize=14)
ax3.grid()

# legends
method_legend = [Line2D([0], [0], color=c, lw=4, label=m) for m, c in col.items()] \
              + [Line2D([0], [0], color='green', lw=4, label='Rural ref')]
type_legend = [
    Line2D([0], [0], color='black', ls=(0,(3,1)), label='Simulation', **SIM),
    Line2D([0], [0], color='black', ls=(0,(8,2)), label='Observation', **OBS),
    Line2D([0], [0], color='black', label='Bias', **BIAS),
]
legend1 = ax3.legend(handles=method_legend, title="Evaluation method", loc="center left",
                     bbox_to_anchor=(1.02, 0.6), borderaxespad=0., fontsize=14)
ax3.add_artist(legend1)
ax3.legend(handles=type_legend, title="Type", loc="upper left", bbox_to_anchor=(1.02, .25),
           borderaxespad=0., fontsize=14, handlelength=4)
# fig.savefig(f'bias_spread_{city}.png', dpi=300, bbox_inches='tight')


In [ ]:
# Fig. 2, variant E (Steven #42/#43): the original figure (cell 55), with
# - a2a_UCN_uc in magenta instead of red, so the three a2a colors differ more
# - the three identical a2a simulated curves drawn as three adjacent solid lines (a ribbon)
from matplotlib.lines import Line2D

col = {'p2a_MS_dist':'darkred', 'a2a_UCN_lcz':'firebrick', 'a2a_MS_uc':'crimson', 'a2a_UCN_uc':'#c51b7d',
       'p2p_UCN_uc':'orangered', 'p2p_UCN_lcz':'darkorange', 'p2p_MS_uc':'salmon'}
key = {'p2p_UCN_uc':'p2p_un_ucdb', 'p2p_UCN_lcz':'p2p_un_lcz', 'p2p_MS_uc':'p2p_ms_ucdb',
       'a2a_UCN_uc':'urclim_un_ucdb', 'a2a_UCN_lcz':'urclim_un_lcz', 'a2a_MS_uc':'urclim_ms_ucdb',
       'p2a_MS_dist':'blend_ms_ucdb'}
val = lambda m, k: uhis[f'UHI{suffix}_{key[m]}_{k}'].values
a2a = ['a2a_UCN_uc', 'a2a_UCN_lcz', 'a2a_MS_uc']
p2p = ['p2p_UCN_uc', 'p2p_UCN_lcz', 'p2p_MS_uc']
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
LW = 4          # line width of all other lines
LW_STRAND = 3   # width of each of the three adjacent a2a lines

def offset_line(ax, y, shift_pt):
    """Shift a line perpendicular to itself by shift_pt points (on screen), returned in data coordinates.
    Call only after the axes limits are fixed."""
    xy = ax.transData.transform(np.column_stack([np.arange(len(y)), y]))
    d = np.diff(xy, axis=0); d /= np.hypot(d[:,0], d[:,1])[:, None]
    n_seg = np.column_stack([-d[:,1], d[:,0]])                      # segment normals
    n = np.vstack([n_seg[:1], n_seg[:-1] + n_seg[1:], n_seg[-1:]])  # miter direction at vertices
    n /= np.hypot(n[:,0], n[:,1])[:, None]
    cos = np.ones(len(y)); cos[1:-1] = np.clip((n[1:-1] * n_seg[:-1]).sum(1), 0.3, 1)
    xy_off = xy + n * (shift_pt * ax.figure.dpi / 72 / cos)[:, None]
    return ax.transData.inverted().transform(xy_off)

fig = plt.figure(figsize=(18,12))
gs = fig.add_gridspec(2, 2)
ax1 = fig.add_subplot(gs[0, 0])   # top left
ax2 = fig.add_subplot(gs[0, 1])   # top right
ax3 = fig.add_subplot(gs[1, :])   # bottom spans both columns
pos = ax3.get_position()
ax3.set_position([pos.x0 + 0.05, pos.y0, pos.width - 0.20, pos.height])

# simulated (ax1) and observed (ax2) UHI; the other type is faded
for ax, main in ((ax1, 'sim'), (ax2, 'obs')):
    al = lambda k: 1.0 if k == main else 0.2
    for ma, mp in zip(a2a, p2p):
        ax.plot(val(ma, 'obs'), color=col[ma], ls=(0,(5,7)), marker='o', lw=LW, alpha=al('obs'))
        ax.plot(val(mp, 'obs'), color=col[mp], ls=(6,(5,7)), marker='o', lw=LW, alpha=al('obs'))
    ax.plot(val('p2a_MS_dist', 'obs'), color=col['p2a_MS_dist'], ls='--', marker='o', lw=LW, alpha=al('obs'))
    for mp in p2p:
        ax.plot(val(mp, 'sim'), color=col[mp], lw=LW, alpha=al('sim'))
    ax.plot(val('p2a_MS_dist', 'sim'), color=col['p2a_MS_dist'], lw=LW, alpha=al('sim'))
    ax.plot(np.zeros(12), color='green', lw=3, marker='o')
    ax.set_xticks(np.arange(0, 12))
    ax.set_xticklabels(months, fontsize=12)
    ax.grid()
ax1.set_title(f"simulated UHI{suffix}", fontsize=16)
ax2.set_title(f"observed UHI{suffix}", fontsize=16)
ax1.set_ylabel(f"UHI{suffix} [°C]", fontsize=14)

# a2a simulated: three adjacent solid lines (needs fixed limits)
for ax, a in ((ax1, 1.0), (ax2, 0.2)):
    ax.set_xlim(ax.get_xlim()); ax.set_ylim(ax.get_ylim())
    for i, ma in enumerate(a2a):
        xy = offset_line(ax, val(ma, 'sim'), (i - 1) * LW_STRAND)
        ax.plot(xy[:,0], xy[:,1], color=col[ma], lw=LW_STRAND, alpha=a, zorder=10, solid_joinstyle='miter')

# biases
for m, c in col.items():
    ax3.plot(val(m, 'bias'), color=c, lw=LW, ls=(0,(3,1,1,1)))
ax3.axhline(y=0, color='black', lw=2, ls='dotted')
ax3.axvline(x=8, color='black', lw=1.3)
# September bias values next to the vertical line
for m, xo, yo in (('p2p_MS_uc', 8.2, -0.8), ('a2a_UCN_uc', 8.1, -0.225), ('p2a_MS_dist', 8.1, 1.86)):
    ax3.text(xo, yo, f"{val(m, 'bias')[8]:.2f}", color=col[m], fontsize=12, va='bottom')
ax3.set_title(f" UHI{suffix} bias", fontsize=16)
ax3.set_xlabel("Month", fontsize=14)
ax3.set_xticks(np.arange(0, 12))
ax3.set_xticklabels(months, fontsize=12)
ax3.set_ylabel(f"UHI{suffix} bias [°C]", fontsize=14)
ax3.grid()

# legends
method_legend = [Line2D([0], [0], color=c, lw=4, label=m) for m, c in col.items()] \
              + [Line2D([0], [0], color='green', lw=4, label='Rural ref')]
type_legend = [
    Line2D([0], [0], color='black', lw=4, ls='-', label='Simulation'),
    Line2D([0], [0], color='black', lw=4, ls='--', marker='o', label='Observation'),
    Line2D([0], [0], color='black', lw=4, ls=(0,(3,1,1,1)), label='Bias'),
]
legend1 = ax3.legend(handles=method_legend, title="Evaluation method", loc="center left",
                     bbox_to_anchor=(1.02, 0.6), borderaxespad=0., fontsize=14)
ax3.add_artist(legend1)
ax3.legend(handles=type_legend, title="Type", loc="upper left", bbox_to_anchor=(1.02, .25),
           borderaxespad=0., fontsize=14)
# fig.savefig(f'bias_spread_{city}.png', dpi=300, bbox_inches='tight')


In [ ]:
# Fig. 2, variant F (Steven #42/#43): the original figure (cell 55), with
# - a2a_UCN_uc in magenta instead of red, so the three a2a colors differ more
# - simulated: short dashes; the three identical a2a curves alternate their three colors
# - observed: long dashes with markers (as before); bias: solid lines (no overlap there)
from matplotlib.lines import Line2D

col = {'p2a_MS_dist':'darkred', 'a2a_UCN_lcz':'firebrick', 'a2a_MS_uc':'crimson', 'a2a_UCN_uc':'#c51b7d',
       'p2p_UCN_uc':'orangered', 'p2p_UCN_lcz':'darkorange', 'p2p_MS_uc':'salmon'}
key = {'p2p_UCN_uc':'p2p_un_ucdb', 'p2p_UCN_lcz':'p2p_un_lcz', 'p2p_MS_uc':'p2p_ms_ucdb',
       'a2a_UCN_uc':'urclim_un_ucdb', 'a2a_UCN_lcz':'urclim_un_lcz', 'a2a_MS_uc':'urclim_ms_ucdb',
       'p2a_MS_dist':'blend_ms_ucdb'}
val = lambda m, k: uhis[f'UHI{suffix}_{key[m]}_{k}'].values
a2a = ['a2a_UCN_uc', 'a2a_UCN_lcz', 'a2a_MS_uc']
p2p = ['p2p_UCN_uc', 'p2p_UCN_lcz', 'p2p_MS_uc']
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
LW = 4          # line width of all other lines
SIM = dict(lw=LW, solid_capstyle='butt', dash_capstyle='butt')

fig = plt.figure(figsize=(18,12))
gs = fig.add_gridspec(2, 2)
ax1 = fig.add_subplot(gs[0, 0])   # top left
ax2 = fig.add_subplot(gs[0, 1])   # top right
ax3 = fig.add_subplot(gs[1, :])   # bottom spans both columns
pos = ax3.get_position()
ax3.set_position([pos.x0 + 0.05, pos.y0, pos.width - 0.20, pos.height])

# simulated (ax1) and observed (ax2) UHI; the other type is faded
for ax, main in ((ax1, 'sim'), (ax2, 'obs')):
    al = lambda k: 1.0 if k == main else 0.2
    for ma, mp in zip(a2a, p2p):
        ax.plot(val(ma, 'obs'), color=col[ma], ls=(0,(5,7)), marker='o', lw=LW, alpha=al('obs'))
        ax.plot(val(mp, 'obs'), color=col[mp], ls=(6,(5,7)), marker='o', lw=LW, alpha=al('obs'))
    ax.plot(val('p2a_MS_dist', 'obs'), color=col['p2a_MS_dist'], ls='--', marker='o', lw=LW, alpha=al('obs'))
    for mp in p2p:
        ax.plot(val(mp, 'sim'), color=col[mp], ls=(0,(3,1)), alpha=al('sim'), **SIM)
    ax.plot(val('p2a_MS_dist', 'sim'), color=col['p2a_MS_dist'], ls=(0,(3,1)), alpha=al('sim'), **SIM)
    for i, ma in enumerate(a2a):
        ax.plot(val(ma, 'sim'), color=col[ma], ls=(4*i,(3,9)), alpha=al('sim'), zorder=10, **SIM)
    ax.plot(np.zeros(12), color='green', lw=3, marker='o')
    ax.set_xticks(np.arange(0, 12))
    ax.set_xticklabels(months, fontsize=12)
    ax.grid()
ax1.set_title(f"simulated UHI{suffix}", fontsize=16)
ax2.set_title(f"observed UHI{suffix}", fontsize=16)
ax1.set_ylabel(f"UHI{suffix} [°C]", fontsize=14)

# biases
for m, c in col.items():
    ax3.plot(val(m, 'bias'), color=c, lw=LW)
ax3.axhline(y=0, color='black', lw=2, ls='dotted')
ax3.axvline(x=8, color='black', lw=1.3)
# September bias values next to the vertical line
for m, xo, yo in (('p2p_MS_uc', 8.2, -0.8), ('a2a_UCN_uc', 8.1, -0.225), ('p2a_MS_dist', 8.1, 1.86)):
    ax3.text(xo, yo, f"{val(m, 'bias')[8]:.2f}", color=col[m], fontsize=12, va='bottom')
ax3.set_title(f" UHI{suffix} bias", fontsize=16)
ax3.set_xlabel("Month", fontsize=14)
ax3.set_xticks(np.arange(0, 12))
ax3.set_xticklabels(months, fontsize=12)
ax3.set_ylabel(f"UHI{suffix} bias [°C]", fontsize=14)
ax3.grid()

# legends
method_legend = [Line2D([0], [0], color=c, lw=4, label=m) for m, c in col.items()] \
              + [Line2D([0], [0], color='green', lw=4, label='Rural ref')]
type_legend = [
    Line2D([0], [0], color='black', ls=(0,(3,1)), label='Simulation', **SIM),
    Line2D([0], [0], color='black', lw=4, ls=(0,(5,7)), marker='o', label='Observation'),
    Line2D([0], [0], color='black', lw=4, ls='-', label='Bias'),
]
legend1 = ax3.legend(handles=method_legend, title="Evaluation method", loc="upper left",
                     bbox_to_anchor=(1.02, 1.0), borderaxespad=0., fontsize=14)   # top aligned with the bias panel
ax3.add_artist(legend1)
# Type legend directly below the method legend
fig.canvas.draw()
bottom = ax3.transAxes.inverted().transform(legend1.get_window_extent())[0, 1]
ax3.legend(handles=type_legend, title="Type", loc="upper left", bbox_to_anchor=(1.02, bottom - 0.03),
           borderaxespad=0., fontsize=14, handlelength=4)
# fig.savefig(f'bias_spread_{city}.png', dpi=300, bbox_inches='tight')


## Method spread
Range (max - min) across the seven methods per month, for simulated, observed UHIN and bias.

In [ ]:
 # calculate the spread of each method for each month 
uhis_df = uhis.to_dataframe()
method_spread_sim, method_spread_obs, method_spread_bias = [], [], []
#also keep the min and max values for each method for each month 
for month in range(12):
    method_spread_sim.append(uhis_df.filter(like='sim').iloc[month].max()-uhis_df.filter(like='sim').iloc[month].min())
    method_spread_obs.append(uhis_df.filter(like='obs').iloc[month].max()-uhis_df.filter(like='obs').iloc[month].min())
    method_spread_bias.append(uhis_df.filter(like='bias').iloc[month].max()-uhis_df.filter(like='bias').iloc[month].min())
    
    


In [ ]:
uhis_df.filter(like='bias')

In [ ]:
#print the mean, min and max of the method spread for sim, obs and bias and also give the month in which the max and min occur
print(f"Method spread for simulation: mean={np.mean(method_spread_sim):.2f}, min={np.min(method_spread_sim):.2f} (month={np.argmin(method_spread_sim)+1}), max={np.max(method_spread_sim):.2f} (month={np.argmax(method_spread_sim)+1}), min {uhis_df.filter(like='sim').min().min()} (month = {np.argmin(uhis_df.filter(like='sim').min()+1 )}), max {uhis_df.filter(like='sim').max().max()} (month = {np.argmax(uhis_df.filter(like='sim').max()+1 )})")
print(f"Method spread for observation: mean={np.mean(method_spread_obs):.2f}, min={np.min(method_spread_obs):.2f} (month={np.argmin(method_spread_obs)+1}), max={np.max(method_spread_obs):.2f} (month={np.argmax(method_spread_obs)+1}), min {uhis_df.filter(like='obs').min().min()} (month = {np.argmin(uhis_df.filter(like='obs').min()+1 )}), max {uhis_df.filter(like='obs').max().max()} (month = {np.argmax(uhis_df.filter(like='obs').max()+1 )})")
print(f"Method spread for bias: mean={np.mean(method_spread_bias):.2f}, min={np.min(method_spread_bias):.2f} (month={np.argmin(method_spread_bias)+1}), max={np.max(method_spread_bias):.2f} (month={np.argmax(method_spread_bias)+1}), min {uhis_df.filter(like='bias').min().min()} (month = {np.argmin(uhis_df.filter(like='bias').min()+1 )}) , max {uhis_df.filter(like='bias').max().max()} (month = {np.argmax(uhis_df.filter(like='bias').max()+1 )})")

In [ ]:
uhis_df = uhis.to_dataframe()
method_spread_sim, method_spread_obs, method_spread_bias = [], [], []

for month in range(12):
    method_spread_sim.append(uhis_df.filter(like='sim').iloc[month].max()-uhis_df.filter(like='sim').iloc[month].min())
    method_spread_obs.append(uhis_df.filter(like='obs').iloc[month].max()-uhis_df.filter(like='obs').iloc[month].min())
    method_spread_bias.append(uhis_df.filter(like='bias').iloc[month].max()-uhis_df.filter(like='bias').iloc[month].min())
uhis_df['method_spread_sim'] = method_spread_sim
uhis_df['method_spread_obs'] = method_spread_obs
uhis_df['method_spread_bias'] = method_spread_bias

method_spread_sim

In [ ]:
# calculate relative method spread for each month by dividing the method spread by the mean of the absolute values of the simulations for that month
print(f'relative spread for simulations')
print(np.array(method_spread_sim) / uhis_df.filter(like='sim').abs().mean(axis=1).values)
print(f'relative spread for observations')
print(np.array(method_spread_obs) / uhis_df.filter(like='obs').abs().mean(axis=1).values)

In [ ]:

fig, ax = plt.subplots(figsize=(15, 7))


ax.plot(uhis[f'UHI{suffix}_urclim_un_ucdb_bias'], label=f'UHI{suffix}_urclim_un_ucdb_bias', color='red', lw=4)
ax.plot(uhis[f'UHI{suffix}_urclim_un_lcz_bias'], label=f'UHI{suffix}_urclim_un_lcz_bias', color='firebrick', lw=4)
ax.plot(uhis[f'UHI{suffix}_urclim_ms_ucdb_bias'], label=f'UHI{suffix}_urclim_ms_ucdb_bias', color='crimson', lw=4)
ax.plot(uhis[f'UHI{suffix}_p2p_un_ucdb_bias'], label=f'UHI{suffix}_p2p_un_ucdb_bias', color='orangered', lw=4)
ax.plot(uhis[f'UHI{suffix}_p2p_un_lcz_bias'], label=f'UHI{suffix}_p2p_un_lcz_bias', color='darkorange', lw=4)
ax.plot(uhis[f'UHI{suffix}_p2p_ms_ucdb_bias'], label=f'UHI{suffix}_p2p_ms_ucdb_bias', color='salmon', lw=4)
ax.plot(uhis[f'UHI{suffix}_blend_ms_ucdb_bias'], label=f'UHI{suffix}_blend_ms_ucdb_bias', color='darkred', lw=4)


ax.plot(np.zeros(12), color='black', lw=3)
ax.set_title(f"  {city}: UHI{suffix} bias", fontsize=16)

ax.set_xlabel("Month", fontsize=14)
#set x ticks to show month names
ax.set_xticks(np.arange(0, 12))
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'], fontsize=12)

ax.set_ylabel(f"UHI{suffix} bias [°C]", fontsize=14)
ax.grid()


In [ ]:

fig, ax = plt.subplots(figsize=(8, 7))


ax.plot(uhis[f'UHI{suffix}_urclim_un_ucdb_obs'], label='urclim_un_ucdb_obs', color='red',  linestyle=(0,(5,7)), marker='o', lw=4, alpha=0.2)
ax.plot(uhis[f'UHI{suffix}_urclim_un_lcz_obs'], label='urclim_un_lcz_obs', color='firebrick', linestyle=(0,(5,7)), marker='o', lw=4, alpha=0.2)
ax.plot(uhis[f'UHI{suffix}_urclim_ms_ucdb_obs'], label='urclim_ms_ucdb_obs', color='crimson', linestyle=(0,(5,7)), marker='o', lw=4, alpha=0.2)


ax.plot(uhis[f'UHI{suffix}_p2p_un_ucdb_obs'], label='urclim_un_ucdb_obs', color='orangered', linestyle=(6,(5,7)), marker='o', lw=4, alpha=0.2)
ax.plot(uhis[f'UHI{suffix}_p2p_un_lcz_obs'], label='urclim_un_lcz_obs', color='darkorange', linestyle=(6,(5,7)), marker='o', lw=4, alpha=0.2)
ax.plot(uhis[f'UHI{suffix}_p2p_ms_ucdb_obs'], label='urclim_ms_ucdb_obs', color='salmon', linestyle=(6,(5,7)), marker='o', lw=4, alpha=0.2)

ax.plot(uhis[f'UHI{suffix}_urclim_un_ucdb_sim'],label = 'urclim_un_ucdb_sim' , color='red', linestyle=(0,(5,10)), lw=4)
ax.plot(uhis[f'UHI{suffix}_urclim_un_lcz_sim'],label = 'urclim_un_lcz_sim', color='firebrick', linestyle=(5,(5,10)), lw=4)
ax.plot(uhis[f'UHI{suffix}_urclim_ms_ucdb_sim'],label = 'urclim_ms_ucdb_sim', color='crimson', linestyle=(10,(5,10)), lw=4)


ax.plot(uhis[f'UHI{suffix}_p2p_un_ucdb_sim'], label='p2p_un_ucdb_sim', color='orangered', lw=4)
ax.plot(uhis[f'UHI{suffix}_p2p_un_lcz_sim'], label='p2p_un_lcz_sim', color='darkorange', lw=4)
ax.plot(uhis[f'UHI{suffix}_p2p_ms_ucdb_sim'], label='p2p_ms_ucdb_sim', color='salmon', lw=4)

ax.plot(uhis[f'UHI{suffix}_blend_ms_ucdb_sim'], label='blend_ms_ucdb_sim', color='darkred', lw=4)
ax.plot(uhis[f'UHI{suffix}_blend_ms_ucdb_obs'], label='blend_ms_ucdb_obs', color='darkred', linestyle='--', marker='o', lw=4,  alpha=0.2)


ax.plot(np.zeros(12), color='green', lw=3,marker='o', label= 'rural_ref')



ax.set_title(f"  {city} \n simulation UHI{suffix} ", fontsize=16)
ax.set_xlabel("Month", fontsize=14)
#set x ticks to show month names
ax.set_xticks(np.arange(0, 12))
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'], fontsize=12)

ax.set_ylabel(f"UHI{suffix} [°C]", fontsize=14)
ax.grid()

# Hardcoded method legend
method_legend = [

    Line2D([0], [0], color='darkred', lw=4, label='blend_ms_ucdb'),

    Line2D([0], [0], color='firebrick', lw=4, label='urclim_un_lcz'),
    Line2D([0], [0], color='crimson', lw=4, label='urclim_ms_ucdb'),
    Line2D([0], [0], color='red', lw=4, label='urclim_un_ucdb'),

    Line2D([0], [0], color='orangered', lw=4, label='p2p_un_ucdb'),
    Line2D([0], [0], color='darkorange', lw=4, label='p2p_un_lcz'),
    Line2D([0], [0], color='salmon', lw=4, label='p2p_ms_ucdb'),
    Line2D([0], [0], color='green', lw=4, label='Rural ref'),

]

# Hardcoded line style legend
type_legend = [
    Line2D([0], [0], color='black', lw=4, linestyle='--',marker='o', label='Observation'),
    Line2D([0], [0], color='black', lw=4, linestyle='-', label='Simulation')
]

# Add both legends
legend1 = plt.legend(handles=method_legend, title='Method', loc='upper left')
plt.gca().add_artist(legend1)
plt.legend(handles=type_legend, title='Type', loc='upper left', bbox_to_anchor=(0.15,1))
fig.show()

In [ ]:

fig, ax = plt.subplots(figsize=(8, 7))


ax.plot(uhis[f'UHI{suffix}_urclim_un_ucdb_obs'], label='urclim_un_ucdb_obs', color='red',  linestyle=(0,(5,7)), marker='o', lw=4)
ax.plot(uhis[f'UHI{suffix}_urclim_un_lcz_obs'], label='urclim_un_lcz_obs', color='firebrick', linestyle=(0,(5,7)), marker='o', lw=4)
ax.plot(uhis[f'UHI{suffix}_urclim_ms_ucdb_obs'], label='urclim_ms_ucdb_obs', color='crimson', linestyle=(0,(5,7)), marker='o', lw=4)


ax.plot(uhis[f'UHI{suffix}_p2p_un_ucdb_obs'], label='urclim_un_ucdb_obs', color='orangered', linestyle=(6,(5,7)), marker='o', lw=4)
ax.plot(uhis[f'UHI{suffix}_p2p_un_lcz_obs'], label='urclim_un_lcz_obs', color='darkorange', linestyle=(6,(5,7)), marker='o', lw=4)
ax.plot(uhis[f'UHI{suffix}_p2p_ms_ucdb_obs'], label='urclim_ms_ucdb_obs', color='salmon', linestyle=(6,(5,7)), marker='o', lw=4)

ax.plot(uhis[f'UHI{suffix}_urclim_un_ucdb_sim'],label = 'urclim_un_ucdb_sim' , color='red', linestyle=(0,(5,10)), lw=4, alpha=0.2)
ax.plot(uhis[f'UHI{suffix}_urclim_un_lcz_sim'],label = 'urclim_un_lcz_sim', color='firebrick', linestyle=(5,(5,10)), lw=4, alpha=0.2)
ax.plot(uhis[f'UHI{suffix}_urclim_ms_ucdb_sim'],label = 'urclim_ms_ucdb_sim', color='crimson', linestyle=(10,(5,10)), lw=4, alpha=0.2)


ax.plot(uhis[f'UHI{suffix}_p2p_un_ucdb_sim'], label='p2p_un_ucdb_sim', color='orangered', lw=4, alpha=0.2)
ax.plot(uhis[f'UHI{suffix}_p2p_un_lcz_sim'], label='p2p_un_lcz_sim', color='darkorange', lw=4, alpha=0.2)
ax.plot(uhis[f'UHI{suffix}_p2p_ms_ucdb_sim'], label='p2p_ms_ucdb_sim', color='salmon', lw=4, alpha=0.2)

ax.plot(uhis[f'UHI{suffix}_blend_ms_ucdb_sim'], label='blend_ms_ucdb_sim', color='darkred', lw=4, alpha=0.2)
ax.plot(uhis[f'UHI{suffix}_blend_ms_ucdb_obs'], label='blend_ms_ucdb_obs', color='darkred', linestyle='--', marker='o', lw=4)


ax.plot(np.zeros(12), color='green', lw=3,marker='o', label= 'rural_ref')


ax.set_title(f"  {city} \n simulation UHI{suffix} ", fontsize=16)
ax.set_xlabel("Month", fontsize=14)
#set x ticks to show month names
ax.set_xticks(np.arange(0, 12))
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'], fontsize=12)
ax.set_ylabel(f"UHI{suffix} [°C]", fontsize=14)
ax.grid()

# Hardcoded method legend
method_legend = [

    Line2D([0], [0], color='darkred', lw=4, label='blend_ms_ucdb'),

    Line2D([0], [0], color='firebrick', lw=4, label='urclim_un_lcz'),
    Line2D([0], [0], color='crimson', lw=4, label='urclim_ms_ucdb'),
    Line2D([0], [0], color='red', lw=4, label='urclim_un_ucdb'),

    Line2D([0], [0], color='orangered', lw=4, label='p2p_un_ucdb'),
    Line2D([0], [0], color='darkorange', lw=4, label='p2p_un_lcz'),
    Line2D([0], [0], color='salmon', lw=4, label='p2p_ms_ucdb'),
    Line2D([0], [0], color='green', lw=4, label='Rural ref'),

]

# Hardcoded line style legend
type_legend = [
    Line2D([0], [0], color='black', lw=4, linestyle='--',marker='o', label='Observation'),
    Line2D([0], [0], color='black', lw=4, linestyle='-', label='Simulation')
]

# Add both legends
legend1 = plt.legend(handles=method_legend, title='Method', loc='upper left')
plt.gca().add_artist(legend1)
plt.legend(handles=type_legend, title='Type', loc='upper left', bbox_to_anchor=(0.15,1))

fig.show()